# Minimal example
Run the size mitigation on a handful of spws in one call and show the final mitigation table.

Requires `smart_size_mitigation_heuristics.py` in the same directory (or on PYTHONPATH).

In [1]:
from smart_size_mitigation_heuristics import (
    build_mitigation_table,
    sizes_from_imlist,
    Limits,
    plan,
)

# 1. Define the input parameters (example case)

In [2]:
spws = [
    {  # representative spw: nbin-eligible, single pointing, robust=2.0
        "spw": "25", "nchan": 3840, "ratio_satisfied": False,
        "pb": 0.9, "is_mosaic": False, "ppb": 4.0, "robust": 2.0,
        "n_roi": 3000, "n_total": 3840,
    },
    {  # smaller nchan, only eligible for nbin if ratio condition met
        "spw": "27", "nchan": 960, "ratio_satisfied": True,
        "pb": 1.0, "is_mosaic": False, "ppb": 3.5, "robust": None,
        "n_roi": 800, "n_total": 960,
    },
    {  # nbin-eligible (nchan=7680), single pointing
        "spw": "29", "nchan": 7680, "ratio_satisfied": False,
        "pb": 0.8, "is_mosaic": False, "ppb": 4.0, "robust": None,
        "n_roi": 1000, "n_total": 7680,
    },
    {  # not nbin-eligible at all (ratio condition not met), tight PB already
        "spw": "31", "nchan": 1920, "ratio_satisfied": False,
        "pb": 0.65, "is_mosaic": False, "ppb": 5.0, "robust": 2.0,
        "n_roi": 1200, "n_total": 1920,
    },
]

# The sizes are not typed in: they are computed from the imaging target list
# (in the pipeline: the targets of makeimlist's result) with calculate_sizes().
# Here: 3 fields, each imaged in every spw.
imsizes = {"25": 990, "27": 1980, "29": 1510, "31": 1400}
imlist = [
    {"field": field, "spw": spw, "imsize": (n, n), "nbin": -1, "specmode": "cube"}
    for field in ("f1", "f2", "f3")
    for spw, n in imsizes.items()
]
get_nchan = {row["spw"]: row["nchan"] for row in spws}.__getitem__  # spw -> nchan

# 2. Compute the sizes and build the per-lever mitigation table

In [3]:
cube_sizes, product_sizes, nfields = sizes_from_imlist(imlist, get_nchan)
print("cube sizes (GB):   ", {k: round(v, 1) for k, v in cube_sizes.items()})
print("product sizes (GB):", {k: round(v, 1) for k, v in product_sizes.items()})
print("nfields:", nfields)

table = build_mitigation_table(spws, use_roi=False, nfields=nfields)
table

cube sizes (GB):    {'25': 15.1, '27': 15.1, '29': 70.0, '31': 15.1}
product sizes (GB): {'25': 90.3, '27': 90.4, '29': 420.3, '31': 90.4}
nfields: 3


,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_reduction
spw,,,,,,,,
25,0.5,0.395062,0.339844,0.0,3,3,0.666667,0.933441
27,0.5,0.510000,0.265306,0.0,3,3,0.666667,0.940000
29,0.5,0.234375,0.437500,0.0,3,3,0.666667,0.928223
31,0.0,0.000000,0.577500,0.0,3,3,0.666667,0.859167


# 3. Set the size limits (GB)

In [4]:
limits = Limits(maxcubesize=10.0, maxproductsize=60.0, maxcubelimit=50.0)
limits


Limits(maxcubesize=10.0, maxproductsize=60.0, maxcubelimit=50.0)

# 4. Run the mitigation in one call. spw "29" is the representative spw here

In [5]:
final, msg = plan(
    table,
    limits,
    imlist,
    get_nchan,
    repr_spw="29",
)

# 5. Display the outcome.

In [6]:
print(f"Result: {msg['shortmsg']}")
print("\nLever scores (product-weighted mitigation, best first):")
print(msg["lever_scores"].round(3).to_string())
print(f"\nSpws kept: {list(final.index[final['kept']])}")

Result: Size was mitigated

Lever scores (product-weighted mitigation, best first):
nfields    0.667
n_bin      0.435
cell       0.421
PB         0.261
ROI        0.000

Spws kept: ['25', '27', '29', '31']


In [7]:
final

,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_reduction,cube_size_GB,product_size_GB,required_reduction,min_steps,levers_used,achieved_reduction,field_reduction,mitigated_cube_GB,mitigated_product_GB,kept
spw,,,,,,,,,,,,,,,,,,
25,0.5,0.395062,0.339844,0.0,3,1,0.666667,0.933441,15.054336,90.349538,0.335740,3,"n_bin,nfields,cell",0.669922,0.666667,4.969107,9.940802,True
27,0.5,0.510000,0.265306,0.0,3,1,0.666667,0.940000,15.054336,90.420106,0.335740,4,"PB,nfields,n_bin,cell",0.820000,0.666667,2.709780,5.425206,True
29,0.5,0.234375,0.437500,0.0,3,1,0.666667,0.928223,70.044672,420.322754,0.857234,4,"n_bin,cell,PB,nfields",0.784668,0.666667,15.082862,30.169651,True
31,0.0,0.000000,0.577500,0.0,3,1,0.666667,0.859167,15.052800,90.363840,0.335672,2,"cell,nfields",0.577500,0.666667,6.359808,12.726241,True
